# 04B — Exportación de Spark a Parquet en Google Colab

**Proyecto:** EdTech Learning Analytics  
**Origen:** `datos/procesados/edtech_dataset_limpio.csv`  
**Salida:** `datos/procesados/edtech_dataset_spark.parquet/`

Este notebook documenta cómo se completó la exportación con PySpark en Colab tras un error de Hadoop en Windows. Complementa al notebook 04.

**Ejecución:** las celdas de las secciones 3–9 están preparadas para Google Colab.


## 1. Problema en Windows

En VS Code, Spark cargó y procesó los datos, pero falló al guardarlos en Parquet por la falta de `C:\hadoop\bin\winutils.exe`.

```python
df_spark.write.mode("overwrite").parquet(str(ruta_parquet))
```

El problema afectó la escritura en esa instalación de Windows, no el procesamiento de Spark.


## 2. Solución: Google Colab

Se utilizó Colab (Linux) con Spark en modo local para:

1. Instalar PySpark y subir el CSV limpio.
2. Leer, analizar y exportar los datos a Parquet.
3. Validar la salida y descargarla en ZIP.
4. Integrar la carpeta exportada en VS Code.

**Spark** procesa los datos; **Parquet** es el formato de almacenamiento.


## 3. Preparación de Spark

Se instala PySpark y se crea una sesión local en Colab. Las versiones se muestran como referencia.


In [ ]:
%pip -q install "pyspark>=4.0,<4.2" "pandas>=2.0,<3.0"

In [ ]:
import sys
import shutil
from pathlib import Path
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, count

spark = (
    SparkSession.builder
    .appName("EdTechLearningAnalytics_Exportacion_Colab")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
print("Python:", sys.version.split()[0])
print("PySpark:", pyspark.__version__)
print("Spark:", spark.version)

## 4. Carga del CSV

Se sube `edtech_dataset_limpio.csv` desde Windows. Colab utiliza la ruta temporal `/content/`.


In [ ]:
from google.colab import files

DATA_PROCESSED = Path("/content/datos/procesados")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ruta_csv = DATA_PROCESSED / "edtech_dataset_limpio.csv"
ruta_parquet = DATA_PROCESSED / "edtech_dataset_spark.parquet"

if not ruta_csv.exists():
    print("Sube edtech_dataset_limpio.csv")
    archivos = files.upload()
    if ruta_csv.name not in archivos:
        raise FileNotFoundError(f"No se subió {ruta_csv.name}")
    ruta_csv.write_bytes(archivos[ruta_csv.name])

print("Archivo de entrada:", ruta_csv)
print("Tamaño (bytes):", ruta_csv.stat().st_size)

## 5. Lectura y revisión de datos

Spark lee el CSV con encabezados e infiere los tipos de columnas. Se comprueban filas, columnas y esquema.

**Referencia del proyecto:** 200,000 registros y 36 columnas.


In [ ]:
df_spark = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(ruta_csv))
)
filas_entrada = df_spark.count()
columnas_entrada = len(df_spark.columns)
print(f"Registros de entrada: {filas_entrada:,}")
print("Columnas de entrada:", columnas_entrada)
df_spark.printSchema()
df_spark.show(5, truncate=False)

# Referencia del notebook 04.
if (filas_entrada, columnas_entrada) != (200000, 36):
    print("Aviso: dimensiones diferentes a la referencia del notebook original.")

## 6. Procesamiento con Spark

Se muestran un filtro, un resumen por curso y una consulta Spark SQL. La exportación conserva el DataFrame completo (`df_spark`).


In [ ]:
df_bajo_promedio = df_spark.filter(
    col("score").isNotNull()
    & (col("score") < col("average_student_score"))
)
print("Registros con score inferior al promedio del alumno:", df_bajo_promedio.count())

rendimiento_por_curso = (
    df_spark.groupBy("course_id", "course_title")
    .agg(
        avg("score").alias("average_score"),
        avg("completion_rate").alias("average_completion_rate"),
        count("*").alias("total_interactions")
    )
    .orderBy(col("average_score").desc())
)
rendimiento_por_curso.show(10, truncate=False)

In [ ]:
df_spark.createOrReplaceTempView("edtech_interactions")
resumen_sql = spark.sql("""
    SELECT course_id, course_title,
           COUNT(*) AS total_interactions,
           AVG(score) AS average_score
    FROM edtech_interactions
    GROUP BY course_id, course_title
    ORDER BY average_score DESC
""")
resumen_sql.show(10, truncate=False)

## 7. Exportación a Parquet

`overwrite` reemplaza la salida existente. Spark genera una carpeta con uno o varios archivos `part-*.parquet`.


In [ ]:
df_spark.write.mode("overwrite").parquet(str(ruta_parquet))

print("Directorio Parquet:", ruta_parquet)
for archivo in sorted(ruta_parquet.iterdir()):
    print(" -", archivo.name)

partes = list(ruta_parquet.glob("part-*.parquet"))
assert partes, "No se encontraron archivos part-*.parquet"
print("Particiones Parquet encontradas:", len(partes))

## 8. Validación

Se vuelve a leer el Parquet y se comparan filas y columnas con el CSV de origen.


In [ ]:
df_parquet = spark.read.parquet(str(ruta_parquet))
filas_salida = df_parquet.count()
columnas_salida = len(df_parquet.columns)

print("Filas CSV:", filas_entrada, "| Filas Parquet:", filas_salida)
print("Columnas CSV:", columnas_entrada, "| Columnas Parquet:", columnas_salida)

assert filas_entrada == filas_salida, "Diferencia en el total de filas"
assert df_spark.columns == df_parquet.columns, "Diferencia en columnas u orden"
print("VALIDACIÓN SATISFACTORIA: filas y columnas coinciden")
df_parquet.show(5, truncate=False)

## 9. Descarga del resultado

Se comprime la carpeta Parquet para descargarla completa desde Colab.


In [ ]:
archivo_zip = shutil.make_archive(
    "/content/edtech_dataset_spark_parquet",
    "zip",
    root_dir=str(DATA_PROCESSED),
    base_dir=ruta_parquet.name
)
print("ZIP preparado:", archivo_zip)
files.download(archivo_zip)

## 10. Integración en VS Code

Descomprime el ZIP y coloca `edtech_dataset_spark.parquet/` dentro de `datos/procesados/`. Si ya existe, respalda o reemplaza la carpeta; evita duplicar niveles.

```text
datos/
└── procesados/
    ├── edtech_dataset_limpio.csv
    └── edtech_dataset_spark.parquet/
        ├── _SUCCESS
        └── part-00000-....snappy.parquet
```

Las rutas `/content/` y `google.colab.files` son específicas de Colab.


## 11. Validación opcional en Windows

Desde `notebooks/`, puedes leer el Parquet con Pandas y PyArrow, sin iniciar Spark. Si falta PyArrow: `pip install pyarrow`.

```python
from pathlib import Path
import pandas as pd

ruta = Path("../datos/procesados/edtech_dataset_spark.parquet")
df = pd.read_parquet(ruta)
print("Filas:", len(df), "Columnas:", len(df.columns))
print(df.head())
```


## 12. Conclusión

La escritura en Windows falló por `winutils.exe`. En Google Colab se generó el Parquet con PySpark, se validaron filas y columnas y se descargó el resultado para integrarlo al proyecto.

**Evidencias:** error inicial, versiones de Spark, conteos de registros y carpeta Parquet generada.

**Para exponer:** Spark procesó el CSV limpio y Colab permitió completar su almacenamiento en formato Parquet desde Linux. El archivo se incorporó después al proyecto en VS Code.
